In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd

TASK_DIR = Path.cwd().parent
INPUT_DIR = TASK_DIR / "input"
OUTPUT_DIR = TASK_DIR / "output"

INPUT_PATH = INPUT_DIR / "adjusted_productivity.csv"
OUTPUT_PATH = OUTPUT_DIR / "adjusted_productivity_imported.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
df_raw = pd.read_csv(INPUT_PATH)
df = df_raw.copy()

display(df.head())

,contribs,contribs_with_corr,current,dblp,department,facultyName,first_asst_job_rank,first_asst_job_year,has_postdoc,is_female,...,phd_rank,phd_year,place,pubs,pubs_adj,pubs_with_corr,pubs_with_corr_adj,recordDate,year,CareerAge
0,0.342857,0.342857,Associate Professor,=Ccedil=etintemel:Ugur,Computer Science,Ugur Cetintemel,21.92,2001,False,False,...,28.65,2001.0,Brown University,2,3.902057,2,3.902057,6/6/11,1998,-3
1,0.250000,0.250000,Associate Professor,=Ccedil=etintemel:Ugur,Computer Science,Ugur Cetintemel,21.92,2001,False,False,...,28.65,2001.0,Brown University,1,1.875160,1,1.875160,6/6/11,1999,-2
2,1.833333,1.833333,Associate Professor,=Ccedil=etintemel:Ugur,Computer Science,Ugur Cetintemel,21.92,2001,False,False,...,28.65,2001.0,Brown University,4,7.214900,4,7.214900,6/6/11,2000,-1
3,1.583333,1.583333,Associate Professor,=Ccedil=etintemel:Ugur,Computer Science,Ugur Cetintemel,21.92,2001,False,False,...,28.65,2001.0,Brown University,3,5.209137,3,5.209137,6/6/11,2001,0
4,1.444444,1.444444,Associate Professor,=Ccedil=etintemel:Ugur,Computer Science,Ugur Cetintemel,21.92,2001,False,False,...,28.65,2001.0,Brown University,4,6.691238,4,6.691238,6/6/11,2002,1


In [3]:
required_cols = ["dblp", "phd_year", "CareerAge", "pubs_adj"]
missing_cols = sorted(set(required_cols) - set(df.columns))

assert not missing_cols, f"req'd cols missing: {missing_cols}"
assert len(df) > 0, "Input empty"

display(pd.DataFrame({"column": df.columns,"raw_dtype": df.dtypes.astype(str).values,}))

,column,raw_dtype
0,contribs,float64
1,contribs_with_corr,float64
2,current,object
3,dblp,object
4,department,object
5,facultyName,object
6,first_asst_job_rank,float64
7,first_asst_job_year,int64
8,has_postdoc,bool
9,is_female,bool


In [4]:
numeric_cols = ["phd_year", "CareerAge", "pubs_adj"]

raw_numeric = {col: pd.to_numeric(df[col], errors="coerce") for col in numeric_cols}

infinite_counts = {col: int(np.isinf(raw_numeric[col].to_numpy(dtype=float, na_value=np.nan)).sum()) for col in numeric_cols}

coercion_counts = {col: int((df[col].notna() & raw_numeric[col].isna()).sum()) for col in numeric_cols}

df["dblp"] = df["dblp"].astype("string").str.strip()
df.loc[df["dblp"].eq(""), "dblp"] = pd.NA

df["phd_year"] = raw_numeric["phd_year"].replace([np.inf, -np.inf], np.nan).astype("Int64")
df["CareerAge"] = raw_numeric["CareerAge"].replace([np.inf, -np.inf], np.nan).astype("Int64")
df["pubs_adj"] = raw_numeric["pubs_adj"].replace([np.inf, -np.inf], np.nan).astype("Float64")

print("NAs coerced to missing:")
display(pd.Series(coercion_counts, name="coerced_to_missing"))

print("INFs converted to missing:")
display(pd.Series(infinite_counts, name="infinite_to_missing"))

NAs coerced to missing:


phd_year     0
CareerAge    0
pubs_adj     0
Name: coerced_to_missing, dtype: int64

INFs converted to missing:


phd_year     0
CareerAge    0
pubs_adj     0
Name: infinite_to_missing, dtype: int64

In [5]:
df.to_csv(OUTPUT_PATH, index=False)